# 📥 Forex_DNN Historical Data Collection Workbench

This interactive notebook is designed to download, validate, and manage historical market data for the Forex_DNN framework.

### Key Features Included:
1. **Execution Modes**:
   - `DEBUG`: Downloads a fast 5-day subset of data using the local `MockDataProvider` to verify operations without needing MT5.
   - `FULL`: Downloads the complete historical range for selected symbols utilizing your production provider.
2. **Data Integrity Checks**: Displays bars count, missing candle estimations, and validation flags.
3. **Visualization**: Plots downloaded candle trends and volume distributions.

In [ ]:
import os
import sys

# Ensure we can import from the framework root
framework_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if framework_root not in sys.path:
    sys.path.append(framework_root)
print(f"Framework root added to sys.path: {framework_root}")

In [ ]:
import logging
import time
from datetime import datetime, timedelta, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from Configs.path_manager import PathManager
from Collecting_Data.historical_data_collector import (
    HistoricalDataCollector,
    MockDataProvider,
    MT5DataProvider
)

## ⚙️ 1. PARAMETERS & INPUT CONFIGURATION

In [ ]:
# Choose "DEBUG" or "FULL"
MODE = "DEBUG" 

SYMBOL = "EURUSD"
TIMEFRAME = "M5"
PROVIDER = "mock"          # "mock" or "mt5"
OUTPUT_DIR = PathManager.get_relative_path("historical_data")

# Start and End dates
START_DATE = "2026-01-01"
END_DATE = "2026-01-10"     # Auto/Today in FULL mode is supported

## 🚀 2. DOWNLOAD RUNNER

In [ ]:
print(f"Initiating data collection in {MODE} mode...")

# Setup provider
if MODE == "DEBUG" or PROVIDER == "mock":
    provider = MockDataProvider()
    # For debug mode, force a limited range
    start_dt = datetime(2026, 1, 1, tzinfo=timezone.utc)
    end_dt = datetime(2026, 1, 6, tzinfo=timezone.utc)
else:
    provider = MT5DataProvider()
    start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.now(timezone.utc) if END_DATE.lower() == "auto" else datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)

if not provider.connect():
    print("Failed to connect to specified provider, using MockDataProvider fallback.")
    provider = MockDataProvider()
    provider.connect()

# Run collector
collector = HistoricalDataCollector(
    provider=provider,
    output_dir=OUTPUT_DIR if MODE == "FULL" else os.path.join(OUTPUT_DIR, "debug_test"),
    format="parquet",
    chunk_size_days=5 if MODE == "DEBUG" else 180,
    max_workers=1,
    monitor=None
)

try:
    results = collector.download_symbol(SYMBOL, TIMEFRAME, start_dt, end_dt)
    print(f"\n[SUCCESS] Status: {results['status']}")
    print(f"Total Bars Saved: {results['bars']}")
    print(f"Range: {results['start']} to {results['end']}")
finally:
    provider.disconnect()

## 📊 3. DIAGNOSTICS & VISUALIZATION

In [ ]:
# Load downloaded file and verify details
import glob
output_path = OUTPUT_DIR if MODE == "FULL" else os.path.join(OUTPUT_DIR, "debug_test")
file_pattern = os.path.join(output_path, SYMBOL, f"{TIMEFRAME}.parquet")
matching_files = glob.glob(file_pattern)

if not matching_files:
    print("No downloaded data file found to visualize.")
else:
    df = pd.read_parquet(matching_files[0])
    print(f"Loaded downloaded file: {matching_files[0]}")
    print(f"Shape: {df.shape} | Columns: {list(df.columns)}")
    print("\nFirst 5 rows:")
    display(df.head())
    
    # Plot Price Chart
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    ax1.plot(df["Datetime"], df["Close"], color="blue", label="Close Price")
    ax1.set_title(f"[{SYMBOL}] Close Price and Volume Profile")
    ax1.set_ylabel("Price")
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot Volume Chart
    ax2.bar(df["Datetime"], df["TickVolume"], color="gray", alpha=0.7, label="Volume")
    ax2.set_xlabel("Datetime")
    ax2.set_ylabel("Tick Volume")
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()